## Customer Support Ticket Resolution with LangGraph

#### Setup

Run the setup cell below to import the libraries required to build the ticket resolution pipeline.

Two language model instances are initialized: 
- `classifier_llm` with `temperature=0` for deterministic classification
- `responder_llm` with `temperature=0.5` for more natural draft responses

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed.

In [ ]:
import operator
from IPython.display import Image, display
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI

import warnings
warnings.filterwarnings("ignore")

classifier_llm = ChatOpenAI(model="gpt-4o", temperature=0)
responder_llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

In this project, you will build a customer support ticket resolution pipeline using LangGraph. The pipeline will:

1. Classify incoming support tickets as billing or technical
2. Route tickets to a category-specific drafting node
3. Generate a response using a specialized LLM prompt
4. Pause for a human agent to approve or reject the draft
5. Send the approved response and log every action along the way

You will progress through four task groups, starting with the state schema and node functions, then assembling the graph, and finally running the full human-in-the-loop workflow.

### Task Group 1 — State Schema and Node Functions

**Task 1: Defining the Ticket State Schema**

Every LangGraph workflow begins with a typed state schema that defines the data flowing between nodes.

A class named `TicketState` that inherits from `TypedDict` is already provided.

Inside the class, define the following string fields: `ticket_id`, `issue`, `category`, `draft_response`, `final_response`, and `approval_status` (which tracks whether the draft was approved, rejected, or sent).

Additionally, include an `action_log` field that uses `Annotated` with a list type and the `operator.add` reducer to accumulate actions across the workflow.

In [ ]:
# Define the state schema for the pipeline
class TicketState(TypedDict):
    ticket_id: str
    issue: str
    category: str
    draft_response: str
    final_response: str
    approval_status: str
    action_log: Annotated[list, operator.add]

**Task 2: Building the Classification Node**

The first node reads the customer's issue and uses the LLM to classify it as either `"billing"` or `"technical"`. This classification determines which drafting node handles the ticket next.

A function `classify_ticket` that accepts `state` of type `TicketState` and returns a `dict` is already defined.

Inside the function:

- Use `classifier_llm` to classify the ticket `issue` as exactly `'billing'` or `'technical'`, and store the result in `response`.

- Determine the `category` by checking whether `"billing"` appears in the lowercased `response` — if it does, set `category` to `"billing"`; otherwise set it to `"technical"`.

- Return a dictionary with `category` (the determined category) and `action_log` (a single-item list noting the assigned `category`).

In [ ]:
# Define a node that classifies the ticket
def classify_ticket(state: TicketState) -> dict:
    response = classifier_llm.invoke(
        f"Classify this support ticket as exactly 'billing' or 'technical':\n{state['issue']}"
    )
    category = "billing" if "billing" in response.content.lower() else "technical"
    return {"category": category, "action_log": [f"Classified as: {category}"]}

**Task 3: Building the Drafting Nodes**

Once classified, the pipeline routes the ticket to a specialized drafting node. Each node uses a tailored prompt to generate a response appropriate for the category.

Two functions are already defined: `draft_billing_response` and `draft_technical_response`. Both accept `state` of type `TicketState` and return a `dict`.

- Inside `draft_billing_response`, use `responder_llm` to write a concise, helpful response to the billing inquiry based on the `issue`, store it in `response`, and return a dictionary with `draft_response` (the generated content) and `action_log` (a single-item list noting that the billing response was drafted).

- Inside `draft_technical_response`, use `responder_llm` to write a clear, step-by-step technical support response to the `issue`, store it in `response`, and return a dictionary with `draft_response` (the generated content) and `action_log` (a single-item list noting that the technical response was drafted).

In [ ]:
# Define a node that drafts a billing response
def draft_billing_response(state: TicketState) -> dict:
    response = responder_llm.invoke(
        f"Write a concise, helpful response to this billing support inquiry:\n{state['issue']}"
    )
    return {
        "draft_response": response.content,
        "action_log": ["Drafted billing response"],
    }

# Define a node that drafts a technical response
def draft_technical_response(state: TicketState) -> dict:
    response = responder_llm.invoke(
        f"Write a clear, step-by-step technical support response:\n{state['issue']}"
    )
    return {
        "draft_response": response.content,
        "action_log": ["Drafted technical response"],
    }

**Task 4: Building the Routing Function**

Conditional edges in LangGraph use a routing function to decide which node executes next. The function reads the current state and returns the name of the target node.

Define a function `route_by_category` that takes a `state` parameter of type `TicketState` and returns either `"draft_billing"` or `"draft_technical"` using a `Literal` type. 

If the `category` equals `"billing"`, return `"draft_billing"`; otherwise return `"draft_technical"`.

In [ ]:
# Define the routing function
def route_by_category(state: TicketState) -> Literal["draft_billing", "draft_technical"]:
    if state["category"] == "billing":
        return "draft_billing"
    return "draft_technical"

### Task Group 2 — Human-in-the-Loop Nodes

**Task 5: Building the Review Node**

Before any draft is sent to the customer, a human agent must review and approve it. LangGraph's `interrupt` function pauses the workflow and surfaces data to the caller. When the caller resumes with a decision, a `Command` object directs the graph to the next step.

A function named `review_response` that accepts `state` of type `TicketState` and returns a `Command` is already defined.

Inside the function:

- Call `interrupt()` with a dictionary containing `draft` (the `draft_response`), `ticket` (the `issue`), and `instructions` set to `"Type 'approve' to send, or 'reject' to close the ticket."`. Store the returned value in `decision`.

- If `decision` equals `"approve"`, return a `Command` with `goto` set to `"send_response"` and `update` setting `approval_status` to `"approved"`. 

- Otherwise, return a `Command` with `goto` set to `END` and `update` setting `approval_status` to `"rejected"`.

In [ ]:
# Define a node that pauses for human review and routes based on the decision
def review_response(state: TicketState) -> Command:
    decision = interrupt({
        "draft": state["draft_response"],
        "ticket": state["issue"],
        "instructions": "Type 'approve' to send, or 'reject' to close the ticket.",
    })
    if decision == "approve":
        return Command(goto="send_response", update={"approval_status": "approved"})
    return Command(goto=END, update={"approval_status": "rejected"})

**Task 6: Building the Send Node**

The final processing node copies the approved draft into `final_response`, updates the status to `"sent"`, and logs the action.

A function named `send_response` that accepts `state` of type `TicketState` and returns a `dict` is already defined.

Inside the function, return a dictionary with `final_response` (the `draft_response` from the state), `approval_status` set to `"sent"`, and `action_log` (a single-item list noting the response was sent for `ticket_id`).

In [ ]:
# Define a node that sends the approved response
def send_response(state: TicketState) -> dict:
    return {
        "final_response": state["draft_response"],
        "approval_status": "sent",
        "action_log": [f"Response sent for ticket {state['ticket_id']}"],
    }

### Task Group 3 — Assembling the Graph

**Task 7: Registering Nodes**

With all node functions defined, let's construct the graph by adding them as nodes and defining the execution flow.

Create a `StateGraph` instance named `builder` and pass `TicketState` as the state schema. 

Add the following nodes to the `builder`:
- `classify` that uses the `classify_ticket` function
- `draft_billing` that uses the `draft_billing_response` function
- `draft_technical` that uses the `draft_technical_response` function
- `review_response` that uses the `review_response` function
- `send_response` that uses the `send_response` function

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(TicketState)

# Add nodes to the graph
builder.add_node("classify", classify_ticket)
builder.add_node("draft_billing", draft_billing_response)
builder.add_node("draft_technical", draft_technical_response)
builder.add_node("review_response", review_response)
builder.add_node("send_response", send_response)

**Task 8: Wiring the Edges**

Set up the execution flow by adding:
- An edge from `START` to the `classify` node
- A conditional edge from the `classify` node that uses the `route_by_category` function
- An edge from the `draft_billing` node to the `review_response` node
- An edge from the `draft_technical` node to the `review_response` node
- An edge from the `review_response` node to the `send_response` node
- An edge from the `send_response` node to `END`

In [ ]:
# Connect the nodes using edges
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_by_category)
builder.add_edge("draft_billing", "review_response")
builder.add_edge("draft_technical", "review_response")
builder.add_edge("review_response", "send_response")
builder.add_edge("send_response", END)

**Task 9: Compiling the Graph with a Checkpointer**

The `interrupt` function requires a checkpointer so LangGraph can save the workflow state before pausing. `MemorySaver` provides an in-memory checkpointer suitable for development and testing.

Create a `MemorySaver` instance and assign it to `memory`.

Call `compile()` on `builder` with `checkpointer` set to `memory`, and assign the result to `ticket_pipeline`.

In [ ]:
# Initialize an in-memory checkpointer to save graph state
memory = MemorySaver()

# Compile the graph with the checkpointer
ticket_pipeline = builder.compile(checkpointer=memory)

Run the cell below to visualize the compiled graph. The diagram shows each node and the edges connecting them.

In [ ]:
display(Image(ticket_pipeline.get_graph().draw_mermaid_png()))

### Task Group 4 — Running the Pipeline

**Task 10: Invoking the Pipeline**

With the graph compiled, let's send a support ticket through the pipeline. The first invocation will classify the ticket, route it to the correct drafting node, generate a draft, and then pause at the review node because of the `interrupt()`.

A `config` dictionary with a `thread_id` is already provided to track this specific run.

Create an `initial_state` dictionary with `ticket_id` set to `"001"`, `issue` set to `"I was charged twice for my subscription this month."`, and empty values for `category`, `draft_response`, `final_response`, `approval_status`, and `action_log`.

Invoke the `ticket_pipeline` by passing `initial_state` and `config`, and store the result in `paused`.

Uncomment the `print()` statements to inspect the draft response and action log.

In [ ]:
# Set thread ID for state tracking
config = {"configurable": {"thread_id": "ticket-001"}}

# Create the initial state for the pipeline
initial_state = {
    "ticket_id": "001",
    "issue": "I was charged twice for my subscription this month.",
    "category": "",
    "draft_response": "",
    "final_response": "",
    "approval_status": "",
    "action_log": [],
}

# Invoke the pipeline with the initial state
paused = ticket_pipeline.invoke(initial_state, config)

print("Paused for review.")
print("Draft response:", paused["draft_response"][:300])
print("Action log:", paused["action_log"])

**Task 11: Resuming with Approval**

The pipeline is now paused at the review node, waiting for a human decision. To resume, we invoke the pipeline again with a `Command` that supplies the agent's decision.

Invoke the `ticket_pipeline` with a `Command` where `resume` is set to `"approve"`, along with the same `config`. Store the output in `final`.

Uncomment the `print()` statements to view the final response, approval status, and the full action log.

In [ ]:
# Resume the pipeline with an approval decision
final = ticket_pipeline.invoke(Command(resume="approve"), config)

print("Final response:", final["final_response"][:300])
print("Approval status:", final["approval_status"])
print("Full action log:", final["action_log"])

Let's review the complete end-to-end flow. Run the cell below to see a summary of every step the pipeline performed.

In [ ]:
print("=" * 100)
print("       TICKET RESOLUTION PIPELINE — SUMMARY")
print("=" * 100)
print()
print(f"Ticket ID:       {final['ticket_id']}")
print(f"Issue:           {final['issue']}")
print(f"Category:        {final['category']}")
print(f"Approval Status: {final['approval_status']}")
print()
print("─" * 100)
print("ACTION LOG")
print("─" * 100)
for i, entry in enumerate(final["action_log"], 1):
    print(f"  {i}. {entry}")
print()
print("─" * 100)
print("FINAL RESPONSE")
print("─" * 100)
print(final["final_response"][:300])
print()
print("=" * 100)